# SHIVA-LLM — Objective 02 — Stage 7
## Publication-ready results and reproducibility package

**Google Colab:** Python 3, CPU, 12–16 GB RAM recommended.

**Upload these six ZIP files:** Stage 4A, Stage 4A-B, Stage 4A-C, Stage 4B, Stage 5, Stage 6.

This notebook verifies cross-stage consistency and packages **existing, observed experiment outputs** into journal-style tables, figures, a manuscript-ready Markdown/LaTeX results draft, checksums, and an artifact index. It **does not retrain, invent results, optimize on test data, or establish causal learning gains**.

All reported policy rewards are **simulated** under explicit one-step contextual-bandit assumptions. A depth-2 decision tree is **not** sequential reinforcement learning.

In [ ]:
import os, io, json, hashlib, zipfile, pathlib, shutil, sys, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import files
INPUTS={
 '4a':'objective02_stage4a_simulator.zip',
 '4ab':'objective02_stage4ab_robustness_audit.zip',
 '4ac':'objective02_stage4ac_reward_redesign.zip',
 '4b':'objective02_stage4b_policy_optimization.zip',
 '5':'objective02_stage5_comparative_evaluation.zip',
 '6':'objective02_stage6_ablation_analysis.zip'
}
uploaded=files.upload()
missing=[x for x in INPUTS.values() if not pathlib.Path(x).exists()]
assert not missing, f'Missing inputs: {missing}'
Z={k:zipfile.ZipFile(v) for k,v in INPUTS.items()}
assert all(z.testzip() is None for z in Z.values()),'Corrupt input archive'
OUT=pathlib.Path('/content/objective02_stage7_outputs')
OUT.mkdir(parents=True,exist_ok=True)
for d in ['tables','figures','reports','provenance','source_figures']:
    (OUT/d).mkdir(exist_ok=True)
print('Loaded',len(Z),'ZIP archives')

In [ ]:
def read_json(stage,name):return json.loads(Z[stage].read(name))
def read_csv(stage,name):return pd.read_csv(io.BytesIO(Z[stage].read(name)))
s4a=read_json('4a','simulator_spec.json')
s4ab=read_json('4ab','audit_report.json')
s4ac=read_json('4ac','stage4ac_report.json')
s4b=read_json('4b','stage4b_report.json')
s5=read_json('5','stage5_report.json')
s6=read_json('6','stage6_report.json')
model_metrics=read_csv('4a','response_model_metrics.csv')
policy=read_csv('4b','locked_test_policy_comparison.csv')
bootstrap=read_csv('5','paired_learner_bootstrap.csv')
robust=read_csv('5','reward_weight_robustness.csv')
cal=read_csv('5','observational_calibration_by_readiness.csv')
ablation=read_csv('6','paired_learner_ablation_ci.csv')
stress=read_csv('6','readiness_perturbation_stress.csv')
sensitivity=read_csv('4ac','locked_test_sensitivity.csv')
replication=read_csv('6','stage5_replication_check.csv')
assert s4a['method']=='one_step_contextual_bandit_simulator'
assert s4b['frozen_stage4ac_reward']==s4ac['selected_design']
assert s4b['train_only_fit'] and s4b['test_not_used_for_selection']
assert s5['stage4b_replication_pass']
assert s6['policy_retrained'] is False and s6['test_used_for_selection'] is False
assert s5['test_events']==s6['test_events']==s4b['test_events']==32522
assert s5['test_learners']==s6['test_learners']==402
assert replication.absolute_difference.max()<1e-8
assert s5['replication_max_absolute_reward_difference']<1e-8
assert s6['stage5_replication_max_error']<1e-8
assert not s4ac['causal_effects_identified']
assert not s5['real_learning_improvement_established']
print('Cross-stage audit passed. Frozen model:',s4b['selected_policy'])

### 1. Canonical tables

Each table is sourced directly from prior stage outputs. Tables are exported as CSV and LaTeX. LaTeX formatting is for manuscript use and does not imply that the results meet any journal's acceptance criteria.

In [ ]:
def save_table(df,stem,floatfmt='%.5f'):
    df.to_csv(OUT/'tables'/f'{stem}.csv',index=False)
    (OUT/'tables'/f'{stem}.tex').write_text(
        df.to_latex(index=False,escape=True,float_format=lambda v:floatfmt%v),encoding='utf-8')
# Model metrics, as recorded, without recomputation
save_table(model_metrics,'table_01_observational_response_model')
order=['learned','readiness_rule','fixed_hard','fixed_medium','random','fixed_easy']
moderate=policy[(policy.scenario=='moderate') & policy.policy.isin(order)].copy()
moderate['order']=moderate.policy.map({v:i for i,v in enumerate(order)})
moderate=moderate.sort_values('order').drop(columns=['order'])
save_table(moderate[['policy','mean_simulated_reward','mean_simulated_regret',
                      'mean_assumed_correctness','fraction_easy','fraction_medium','fraction_hard']],
           'table_02_moderate_policy_comparison')
save_table(policy[['scenario','policy','mean_simulated_reward','mean_simulated_regret']],
           'table_03_effect_sensitivity')
save_table(bootstrap[['scenario','comparison','mean','ci_95_low','ci_95_high','bootstrap_reps']],
           'table_04_cluster_bootstrap')
save_table(cal,'table_05_observational_calibration')
save_table(ablation,'table_06_reward_component_ablation')
save_table(stress.groupby('perturbation',as_index=False).agg(
    reward_mean=('mean_reward_on_original_context','mean'),
    action_change_fraction=('action_change_fraction','mean')),
    'table_07_readiness_perturbation')
save_table(robust[['reward_variant','scenario','learned_minus_rule',
                   'learned_minus_best_fixed','oracle_minus_learned']],
           'table_08_reward_weight_sensitivity')
save_table(sensitivity,'table_09_stage4ac_action_sensitivity')
print('Created nine tables (CSV and LaTeX)')

### 2. Publication figures

The figures show *simulated* reward comparisons, sensitivity to effect assumptions, and observed-versus-predicted correctness calibration. Figures are saved in PNG and PDF, with vector PDF preferred for typesetting.

In [ ]:
plt.rcParams.update({'font.size':10,'figure.dpi':120,'savefig.dpi':240})
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT/'figures'/f'{name}.png',bbox_inches='tight')
    plt.savefig(OUT/'figures'/f'{name}.pdf',bbox_inches='tight')
    plt.show()
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(moderate))
ax.bar(x,moderate.mean_simulated_reward)
ax.set_xticks(x,[v.replace('_',' ') for v in moderate.policy],rotation=25,ha='right')
ax.set_ylabel('Mean simulated reward')
ax.set_title('Held-out comparison under moderate assumed effect')
savefig('figure_01_policy_rewards')
fig,ax=plt.subplots(figsize=(7,4))
scenario_order=list(s4b['effect_scenarios'])
diff=[]
for scenario in scenario_order:
    part=policy[policy.scenario==scenario]
    learned=float(part.loc[part.policy=='learned','mean_simulated_reward'].iloc[0])
    best=float(part.loc[part.policy.isin(['fixed_easy','fixed_medium','fixed_hard']),
                        'mean_simulated_reward'].max())
    diff.append(learned-best)
ax.bar(scenario_order,diff)
ax.axhline(0,color='black',linewidth=.8)
ax.set_ylabel('Learned minus best fixed simulated reward')
ax.set_title('Sensitivity to assumed action effects')
savefig('figure_02_assumed_effect_sensitivity')
fig,ax=plt.subplots(figsize=(6,4))
xx=np.arange(len(cal))
ax.bar(xx-.18,cal.predicted_correctness,width=.36,label='Predicted')
ax.bar(xx+.18,cal.observed_correctness,width=.36,label='Observed')
ax.set_xticks(xx,cal.readiness)
ax.set_ylabel('Correctness rate')
ax.set_title('Observed response calibration by readiness')
ax.legend()
savefig('figure_03_readiness_calibration')

In [ ]:
# Copy source figures without alteration; preserve provenance.
fig_sources=[
 ('4a','calibration.png'),
 ('4ab','heldout_reliability.png'),
 ('4ac','utility_curves.png'),
 ('4b','test_policy_rewards.png'),
 ('5','stage5_policy_comparisons.png'),
 ('6','stage6_ablation_figures.png')]
for stage,name in fig_sources:
    if name in Z[stage].namelist():
        (OUT/'source_figures'/f'stage{stage}_{name}').write_bytes(Z[stage].read(name))
print('Copied',len(list((OUT/'source_figures').iterdir())),'unaltered source figures')

### 3. Manuscript-ready results and limitations

The narrative below is generated from the actual input tables and audit metadata. It deliberately avoids claims of actual student-learning improvement, validated interventions, or sequential RL.

In [ ]:
def value(policy_name,scenario='moderate'):
    return float(policy[(policy.policy==policy_name)&(policy.scenario==scenario)].mean_simulated_reward.iloc[0])
learner=value('learned')
rule=value('readiness_rule')
fixed=max(value(x) for x in ['fixed_easy','fixed_medium','fixed_hard'])
bmod=bootstrap[(bootstrap.scenario=='moderate')&
               (bootstrap.comparison=='learned_minus_readiness_rule')].iloc[0]
bfix=bootstrap[(bootstrap.scenario=='moderate')&
               (bootstrap.comparison=='learned_minus_best_fixed')].iloc[0]
gap=cal.copy()
gap['abs_gap']=(gap.predicted_correctness-gap.observed_correctness).abs()
worst=gap.loc[gap.abs_gap.idxmax()]
lines=[
 '# Objective 02: Results and discussion (simulation-based)',
 '',
 '## Study setting',
 f"The held-out test partition comprised {s5['test_events']:,} interactions from {s5['test_learners']} learners. "
 f"The frozen policy was a {s4b['selected_policy']} model, fitted on training contexts and selected using validation data. "
 'The environment was a one-step contextual bandit with assumed easy, medium, and hard action effects.',
 '',
 '## Primary simulated comparison',
 f"Under the moderate assumed action effect, the learned policy achieved a mean simulated reward of {learner:.6f}, "
 f"versus {rule:.6f} for the readiness rule and {fixed:.6f} for the best fixed action. "
 f"The learned-minus-readiness difference was {learner-rule:.6f}; the learner-cluster bootstrap "
 f"95% interval was [{bmod.ci_95_low:.6f}, {bmod.ci_95_high:.6f}]. "
 f"The learned-minus-best-fixed difference was {learner-fixed:.6f}, with bootstrap "
 f"95% interval [{bfix.ci_95_low:.6f}, {bfix.ci_95_high:.6f}].",
 '',
 '## Sensitivity',
 f"The learned-minus-best-fixed simulated reward differences across effect scenarios were: "
 + ', '.join(f'{k}={v:+.6f}' for k,v in zip(scenario_order,diff)) + '. '
 'The negative differences under weaker assumptions indicate that superiority is not robust across scenarios.',
 '',
 '## Observational response calibration',
 f"The largest absolute readiness-group calibration gap was in the {worst.readiness} group: "
 f"predicted correctness {worst.predicted_correctness:.4f} versus observed {worst.observed_correctness:.4f}.",
 '',
 '## Ablation interpretation',
 'Reward-component ablations change the utility objective; they are preference-sensitivity tests rather than '
 'empirical component-effect identification. Readiness perturbations stress the single input to the frozen policy '
 'and do not constitute a comparison between alternative knowledge-tracing architectures.',
 '',
 '## Limitations',
 'The historical action labels were not authenticated instructional interventions. '
 'The simulator assumes action-dependent correctness, challenge credit, and effort cost, and does not identify '
 'causal effects or learner-state transitions. Bootstrap confidence intervals condition on those assumptions. '
 'The frozen model is a one-step contextual-bandit policy, not a validated sequential reinforcement-learning agent. '
 'No observed mastery improvement, long-term retention benefit, or real-world instructional effectiveness is established. '
 'Test-set robustness and ablation analyses are post-hoc and should not guide further model selection without a new holdout.',
 '',
 '## Reproducibility',
 f"The maximum Stage 5-to-6 reward replication error was {float(replication.absolute_difference.max()):.3e}. "
 'All input archive hashes and source-file provenance are recorded in the accompanying manifest.'
]
text='\n\n'.join(lines)
(OUT/'reports'/'manuscript_results_and_limitations.md').write_text(text,encoding='utf-8')
print(text[:1600])

### 4. Provenance, hashes, and consistency checks

The manifest records SHA-256 checksums for every input ZIP and every internal member, along with the canonical output files. The final package excludes large raw interaction records and model checkpoints; it is a compact **results package**, not a substitute for the full executable code and raw data provenance.

In [ ]:
def sha256(data):return hashlib.sha256(data).hexdigest()
manifest={'project':'SHIVA-LLM Objective 02','stage':'7',
          'inputs':{},'artifacts':[],
          'software':{'python':sys.version,'pandas':pd.__version__,
                      'numpy':np.__version__,'platform':platform.platform()},
          'audit':{
             'test_events':s5['test_events'],
             'test_learners':s5['test_learners'],
             'frozen_policy':s4b['selected_policy'],
             'stage5_replication_pass':s5['stage4b_replication_pass'],
             'max_stage6_replication_error':float(replication.absolute_difference.max()),
             'causal_effects_identified':False,
             'sequential_rl_validated':False}}
for stage,name in INPUTS.items():
    with open(name,'rb') as f:zip_bytes=f.read()
    manifest['inputs'][stage]={
       'filename':name,'bytes':len(zip_bytes),'sha256':sha256(zip_bytes),
       'members':{member:{'bytes':len(Z[stage].read(member)),
                           'sha256':sha256(Z[stage].read(member))}
                  for member in Z[stage].namelist() if not member.endswith('/')}}
# Save a separate stage audit report
audit={'stage':'7','cross_stage_integrity_passed':True,
       'test_events':s5['test_events'],'test_learners':s5['test_learners'],
       'selected_model':s4b['selected_policy'],
       'selected_reward':s4ac['selected_design'],
       'max_stage5_stage6_difference':float(replication.absolute_difference.max()),
       'real_causal_effect_established':False,
       'real_learning_improvement_established':False,
       'sequential_reinforcement_learning_evaluated':False,
       'posthoc_analyses_used_for_model_selection':False}
(OUT/'reports'/'stage7_audit.json').write_text(json.dumps(audit,indent=2))
for f in sorted(OUT.rglob('*')):
    if f.is_file():
        manifest['artifacts'].append({'path':str(f.relative_to(OUT)),
                                     'bytes':f.stat().st_size,
                                     'sha256':sha256(f.read_bytes())})
(OUT/'provenance'/'manifest.json').write_text(json.dumps(manifest,indent=2))
(OUT/'provenance'/'input_checksums.csv').write_text(
 'stage,filename,sha256,bytes\n'+''.join(
 f"{k},{v['filename']},{v['sha256']},{v['bytes']}\n" for k,v in manifest['inputs'].items()))
print('Manifest written for',len(manifest['inputs']),'input ZIPs and',
      len(manifest['artifacts']),'output files')

In [ ]:
(OUT/'README.md').write_text(
 '# SHIVA-LLM Objective 02 — Stage 7 results package\n\n'
 'This package consolidates previously executed Stage 4A–6 experiments.\n\n'
 'Tables: CSV + LaTeX; Figures: PNG + PDF; Reports: manuscript results and audit; '
 'Provenance: SHA-256 manifest of inputs and generated outputs.\n\n'
 '**Scientific limitation:** simulated one-step contextual-bandit policy optimization; '
 'not causal evidence of real student learning gains or validated sequential RL.\n\n'
 'To reproduce numerical experiments, retain the original stage notebooks, input ZIPs, '
 'HyperKT artifacts and dataset provenance. This package alone is not full computational reproduction.\n',
 encoding='utf-8')
archive=pathlib.Path('/content/objective02_stage7_publication_package.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file():z.write(f,arcname=str(f.relative_to(OUT)))
assert zipfile.ZipFile(archive).testzip() is None
print('Created',archive,'size',archive.stat().st_size,'bytes')
files.download(str(archive))